In [48]:
import os
import pandas as pd
import numpy as np


In [50]:
df=pd.read_csv("clean_stage1.csv")

In [51]:
df.head

<bound method NDFrame.head of          Unnamed: 0    cow        date  hour  IN_ALLEYS      REST       EAT  \
0                 0   6601  2018-10-25    11   1603.585  1355.386    41.024   
1                 1   6601  2018-10-25    12   1586.965   138.501  1874.528   
2                 2   6601  2018-10-25    13   1442.930   567.066  1584.866   
3                 3   6601  2018-10-25    14    209.571  2728.410   662.013   
4                 4   6601  2018-10-25    15   1094.275   298.497  2207.222   
...             ...    ...         ...   ...        ...       ...       ...   
2351340     2351340  90916  2015-12-31    20    733.940   549.540  2316.519   
2351341     2351341  90916  2015-12-31    21      0.000  3599.999     0.000   
2351342     2351342  90916  2015-12-31    22     73.783  3526.216     0.000   
2351343     2351343  90916  2015-12-31    23   1778.370   257.090  1564.539   
2351344     2351344  90916  2015-12-31    24   1156.411  1163.549  1280.039   

         ACTIVITY_LEV

#### List of Cattle with no disease records 

In [52]:
# Group by cow and check if all disease values are 0 across all their rows, and for that if the cow have no diseases in the four diseases which were taken for prediction, then their sum also will be zero 
healthy_cows = df.groupby('cow')[['oestrus', 'calving', 'lameness', 'mastitis']].sum()

# Filter cows with zero in all diseases
completely_healthy_cows = healthy_cows[(healthy_cows == 0).all(axis=1)].index

print("Cows with NO disease in ANY of their records:")
print(completely_healthy_cows.tolist())


Cows with NO disease in ANY of their records:
[151, 153, 162, 173, 189, 1177, 2158, 2165, 2175, 2179, 2185, 2187, 2284, 2399, 2603, 2699, 4262, 5896, 8595, 9195, 9481, 9994, 35687, 45683, 48094, 48492, 48500, 48599, 48642, 49108, 49149, 49230, 49435, 49653, 49688, 49697, 50422, 50433, 50468, 50615, 50759, 50794, 53916, 66242, 66259, 72698, 72713, 72838, 72922, 72953, 73088, 73134, 73141, 74511, 74612, 74674, 74720, 74737, 74768, 74775, 74821, 74908, 74953, 81446, 81453, 81648, 81718, 81725, 81732, 81763, 81857, 81927, 81972, 85764, 85935, 85959, 85973, 86109, 86123, 86217, 90853, 96844, 96851, 96868, 96899, 96907, 96921, 96938, 96952, 96976, 96990, 97018, 97049, 97056, 97087, 97210, 98134, 98158, 98211, 98228, 98235, 98266, 98273, 98280, 98312, 98336, 98343]


In [56]:
# Group by cow and sum disease indicators
disease_sums = df.groupby('cow')[['oestrus', 'calving', 'lameness', 'mastitis']].sum()

# Identify cows with NO disease (all sums are 0)
healthy_cows = disease_sums[(disease_sums == 0).all(axis=1)].index

# Remove all records of healthy cows
no_healthy_cows = df[~df['cow'].isin(healthy_cows)].copy()

# Display shape and preview
print(f"Original shape: {df.shape}")
print(f"Filtered shape (no healthy cows): {no_healthy_cows.shape}")
no_healthy_cows.head()


Original shape: (2351345, 16)
Filtered shape (no healthy cows): (1935667, 16)


,Unnamed: 0,cow,date,hour,IN_ALLEYS,REST,EAT,ACTIVITY_LEVEL,oestrus,calving,lameness,mastitis,other_disease,accidents,disturbance,mixing
0,0,6601,2018-10-25,11,1603.585,1355.386,41.024,-37.93510,0,0,0,0,0,0,0,0
1,1,6601,2018-10-25,12,1586.965,138.501,1874.528,1009.36093,0,0,0,0,0,0,0,0
2,2,6601,2018-10-25,13,1442.930,567.066,1584.866,766.08734,0,0,0,0,0,0,0,0
3,3,6601,2018-10-25,14,209.571,2728.410,662.013,-315.95748,0,0,0,0,0,0,0,0
4,4,6601,2018-10-25,15,1094.275,298.497,2207.222,1033.46293,0,0,0,0,0,0,0,0


#### The cattle with disease conditions alone is taken for the study 

In [58]:
no_healthy_cows.to_csv("unhealthy_cows.csv")

#### Dropping the irrelevant columns in the study 

In [59]:
# Drop irrelevant columns
no_healthy_cows.drop(columns=['other_disease', 'accidents', 'mixing'], inplace=True)

# Confirm the result
print("Remaining columns:", no_healthy_cows.columns.tolist())

Remaining columns: ['Unnamed: 0', 'cow', 'date', 'hour', 'IN_ALLEYS', 'REST', 'EAT', 'ACTIVITY_LEVEL', 'oestrus', 'calving', 'lameness', 'mastitis', 'disturbance']


#### Finding the cattle with no data such as eat, rest and grazing with 0 seconds 

In [60]:

df1= no_healthy_cows
# Filter rows where EAT, REST, and IN_ALLEYS are all 0
zero_activity_rows = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]

# Show result
print(f"Total rows with EAT, REST, and IN_ALLEYS all 0: {len(zero_activity_rows)}")
zero_activity_rows.head(50)


Total rows with EAT, REST, and IN_ALLEYS all 0: 20


,Unnamed: 0,cow,date,hour,IN_ALLEYS,REST,EAT,ACTIVITY_LEVEL,oestrus,calving,lameness,mastitis,disturbance
163657,163657,5098,2013-10-01,14,0.0,0.0,0.0,0.0,0,0,0,0,0
163658,163658,5098,2013-10-01,15,0.0,0.0,0.0,0.0,0,0,0,0,0
163659,163659,5098,2013-10-01,16,0.0,0.0,0.0,0.0,0,0,0,0,0
308118,308118,49382,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
390111,390111,50351,2015-04-11,23,0.0,0.0,0.0,0.0,0,0,0,0,1
457110,457110,50508,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
554607,554607,50738,2015-04-11,23,0.0,0.0,0.0,0.0,0,0,0,0,1
600102,600102,53891,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
626310,626310,53930,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
795894,795894,72775,2015-11-25,14,0.0,0.0,0.0,0.0,0,0,0,1,1


In [61]:
zero_value_rows = (df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)

# Apply backfill to the rows for the selected columns
df1.loc[zero_value_rows, ['EAT', 'REST', 'IN_ALLEYS']] = df1[['EAT', 'REST', 'IN_ALLEYS']].bfill()
#remaininig columns and their count 
remaining = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]
print(f"Remaining rows with all zero activity after backfill: {len(remaining)}")


Remaining rows with all zero activity after backfill: 20


In [129]:
# Check for any remaining rows where all three values are still 0
remaining_zero_rows = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]

# Print the result
if remaining_zero_rows.empty:
    print("All targeted rows have been successfully backfilled.")
else:
    print(f"{len(remaining_zero_rows)} rows still have all zeros in EAT, REST, and IN_ALLEYS.")
    display(remaining_zero_rows.head())


All targeted rows have been successfully backfilled.


In [131]:
# Identify rows with all zeros in EAT, REST, and IN_ALLEYS
zero_rows_mask = (df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)
zero_rows_indices = df1[zero_rows_mask].index

# Create temporary backfilled and forward-filled versions
backfilled = df1[['EAT', 'REST', 'IN_ALLEYS']].bfill()
forwardfilled = df1[['EAT', 'REST', 'IN_ALLEYS']].ffill()

# Choose backfill where possible, otherwise forwardfill
df1.loc[zero_rows_indices, ['EAT', 'REST', 'IN_ALLEYS']] = backfilled.loc[zero_rows_indices].fillna(
    forwardfilled.loc[zero_rows_indices]
)

# Recalculate ACTIVITY_LEVEL for those rows
df1.loc[zero_rows_indices, 'ACTIVITY_LEVEL'] = (
    -0.23 * df1.loc[zero_rows_indices, 'REST'] +
    0.16 * df1.loc[zero_rows_indices, 'IN_ALLEYS'] +
    0.42 * df1.loc[zero_rows_indices, 'EAT']
)

# Confirm the change
final_check = df1.loc[zero_rows_indices]
print(f"Imputed {len(final_check)} rows and recalculated ACTIVITY_LEVEL.")
display(final_check.head())


Imputed 0 rows and recalculated ACTIVITY_LEVEL.


,,Unnamed: 0,cow,date,hour,IN_ALLEYS,REST,EAT,ACTIVITY_LEVEL,oestrus,calving,lameness,mastitis,disturbance
cow,,,,,,,,,,,,,,


In [70]:

df1= no_healthy_cows
# Filter rows where EAT, REST, and IN_ALLEYS are all 0
zero_activity_rows = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]

# Show result
print(f"Total rows with EAT, REST, and IN_ALLEYS all 0: {len(zero_activity_rows)}")
zero_activity_rows.head(50)


Total rows with EAT, REST, and IN_ALLEYS all 0: 20


,Unnamed: 0,cow,date,hour,IN_ALLEYS,REST,EAT,ACTIVITY_LEVEL,oestrus,calving,lameness,mastitis,disturbance
163657,163657,5098,2013-10-01,14,0.0,0.0,0.0,0.0,0,0,0,0,0
163658,163658,5098,2013-10-01,15,0.0,0.0,0.0,0.0,0,0,0,0,0
163659,163659,5098,2013-10-01,16,0.0,0.0,0.0,0.0,0,0,0,0,0
308118,308118,49382,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
390111,390111,50351,2015-04-11,23,0.0,0.0,0.0,0.0,0,0,0,0,1
457110,457110,50508,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
554607,554607,50738,2015-04-11,23,0.0,0.0,0.0,0.0,0,0,0,0,1
600102,600102,53891,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
626310,626310,53930,2014-12-18,14,0.0,0.0,0.0,0.0,0,0,0,0,0
795894,795894,72775,2015-11-25,14,0.0,0.0,0.0,0.0,0,0,0,1,1


In [72]:
# Function to fill zero-activity rows within each cow group
def fill_zero_activity(group):
    # Find zero-activity rows in this group
    zero_mask = (group['EAT'] == 0) & (group['REST'] == 0) & (group['IN_ALLEYS'] == 0)
    
    # Only fill those rows
    
    # Apply backfill then forward fill within group for these columns
    group.loc[zero_mask, ['EAT', 'REST', 'IN_ALLEYS']] = (
        group[['EAT', 'REST', 'IN_ALLEYS']].bfill().ffill().loc[zero_mask]
    )
    
    # Recalculate ACTIVITY_LEVEL for zero rows after imputation
    group.loc[zero_mask, 'ACTIVITY_LEVEL'] = (
        -0.23 * group.loc[zero_mask, 'REST'] +
        0.16 * group.loc[zero_mask, 'IN_ALLEYS'] +
        0.42 * group.loc[zero_mask, 'EAT']
    )
    
    return group

# Apply this function to each cow group
df1 = df1.groupby('cow').apply(fill_zero_activity)

# Check if any zero-activity rows remain
remaining_zero = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]

print(f"Remaining zero-activity rows after grouped filling: {len(remaining_zero)}")


Remaining zero-activity rows after grouped filling: 20


C:\Users\vishn\AppData\Local\Temp\ipykernel_26412\2707783763.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1 = df1.groupby('cow').apply(fill_zero_activity)


In [37]:
remaining_zero = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]
print(remaining_zero[['cow', 'date', 'hour']])


                 cow        date  hour
cow                                   
5098  163657    5098  2013-10-01    14
      163658    5098  2013-10-01    15
      163659    5098  2013-10-01    16
49382 308118   49382  2014-12-18    14
50351 390111   50351  2015-04-11    23
50508 457110   50508  2014-12-18    14
50738 554607   50738  2015-04-11    23
53891 600102   53891  2014-12-18    14
53930 626310   53930  2014-12-18    14
72775 795894   72775  2015-11-25    14
73127 988254   73127  2014-12-18    14
73196 1015983  73196  2015-04-11    23
74991 1348230  74991  2014-12-18    14
81554 1817478  81554  2014-12-18    14
81864 1960998  81864  2014-12-18    14
85795 1453014  85795  2014-12-18    14
86022 1612878  86022  2014-12-18    14
86147 1693974  86147  2015-11-25    14
90891 2323494  90891  2014-12-18    14
97335 2250534  97335  2015-11-25    14


In [44]:
df1 = df1.reset_index(drop=True)  # remove current index (which has 'cow' as a level)



In [74]:
def fill_zeros_with_ffill_bfill(group):
    # Mask for rows where EAT, REST, IN_ALLEYS are all zero
    zero_mask = (group['EAT'] == 0) & (group['REST'] == 0) & (group['IN_ALLEYS'] == 0)

    # Apply forward fill, then backfill only to these zero rows
    # We fill EAT, REST, IN_ALLEYS separately
    group.loc[zero_mask, ['EAT']] = group['EAT'].replace(0, np.nan).ffill().bfill().loc[zero_mask]
    group.loc[zero_mask, ['REST']] = group['REST'].replace(0, np.nan).ffill().bfill().loc[zero_mask]
    group.loc[zero_mask, ['IN_ALLEYS']] = group['IN_ALLEYS'].replace(0, np.nan).ffill().bfill().loc[zero_mask]

    # Recalculate ACTIVITY_LEVEL for these zero rows
    group.loc[zero_mask, 'ACTIVITY_LEVEL'] = (
        -0.23 * group.loc[zero_mask, 'REST'] +
         0.16 * group.loc[zero_mask, 'IN_ALLEYS'] +
         0.42 * group.loc[zero_mask, 'EAT']
    )

    return group

# Reset index to avoid groupby ambiguity if needed
df1 = df1.reset_index(drop=True)

# Apply the function by group
df1 = df1.groupby('cow').apply(fill_zeros_with_ffill_bfill)


C:\Users\vishn\AppData\Local\Temp\ipykernel_26412\1678554402.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df1 = df1.groupby('cow').apply(fill_zeros_with_ffill_bfill)


#### Checking for the successfull imputation

In [76]:

remaining_zero = df1[(df1['EAT'] == 0) & (df1['REST'] == 0) & (df1['IN_ALLEYS'] == 0)]
print(f"Remaining zero-activity rows after mean imputation: {len(remaining_zero)}")

Remaining zero-activity rows after mean imputation: 0


#### Saving the file in csv file 

In [78]:
df1.to_csv("full_data_unhealthy_imputed.csv")

In [80]:
df=pd.read_csv("full_data_unhealthy_imputed.csv")

In [82]:
df.head()

,cow,Unnamed: 1,Unnamed: 0,cow.1,date,hour,IN_ALLEYS,REST,EAT,ACTIVITY_LEVEL,oestrus,calving,lameness,mastitis,disturbance
0,156,0,117745,156,2015-03-02,1,35.614,3564.385,0.0,-814.11031,0,0,0,0,1
1,156,1,117746,156,2015-03-02,2,0.000,3599.999,0.0,-827.99977,0,0,0,0,1
2,156,2,117747,156,2015-03-02,3,0.000,3599.999,0.0,-827.99977,0,0,0,0,1
3,156,3,117748,156,2015-03-02,4,0.000,3599.999,0.0,-827.99977,0,0,0,0,1
4,156,4,117749,156,2015-03-02,5,6.676,3593.323,0.0,-825.39613,0,0,0,0,1


#### Checkiong for the imbalance in data

In [86]:
# Count 0s and 1s for each disease
disease_cols = ['calving', 'lameness', 'mastitis', 'oestrus']

disease_summary = {}

for col in disease_cols:
    counts = df1[col].value_counts().sort_index()  
    disease_summary[col] = {
        'Absent (0)': counts.get(0, 0),
        'Present (1)': counts.get(1, 0)
    }

disease_summary_df = pd.DataFrame(disease_summary).T
print(disease_summary_df)


          Absent (0)  Present (1)
calving      1931371         4296
lameness     1932451         3216
mastitis     1934611         1056
oestrus      1927723         7944


In [88]:
zero_disease_rows = df1[
    (df1['oestrus'] == 0) &
    (df1['lameness'] == 0) &
    (df1['calving'] == 0) &
    (df1['mastitis'] == 0)
]

print(f"Total rows with no diseases (oestrus, lameness, calving, mastitis all 0): {len(zero_disease_rows)}")
zero_disease_rows.head()


Total rows with no diseases (oestrus, lameness, calving, mastitis all 0): 1919227


Unnamed: 0  cow        date  hour  IN_ALLEYS      REST  EAT  \
cow                                                                  
156 0      117745  156  2015-03-02     1     35.614  3564.385  0.0   
    1      117746  156  2015-03-02     2      0.000  3599.999  0.0   
    2      117747  156  2015-03-02     3      0.000  3599.999  0.0   
    3      117748  156  2015-03-02     4      0.000  3599.999  0.0   
    4      117749  156  2015-03-02     5      6.676  3593.323  0.0   

       ACTIVITY_LEVEL  oestrus  calving  lameness  mastitis  disturbance  
cow                                                                       
156 0      -814.11031        0        0         0         0            1  
    1      -827.99977        0        0         0         0            1  
    2      -827.99977        0        0         0         0            1  
    3      -827.99977        0        0         0         0            1  
    4      -825.39613        0        0         0         0            1

#### For the sake of understanding, checkingh the rows with disease conditions 

In [94]:
# Filter rows where at least one disease is present
disease_present_df = df1[
    (df1['oestrus'] == 1) |
    (df1['lameness'] == 1) |
    (df1['calving'] == 1) |
    (df1['mastitis'] == 1)
]


disease_present_df.to_csv("disease_present_condiiton.csv", index=False)

print(len(disease_present_df))


16440


In [92]:
# Count 0s and 1s for each disease
disease_cols = ['calving', 'lameness', 'mastitis', 'oestrus']

disease_summary = {}

for col in disease_cols:
    counts = disease_present_df[col].value_counts().sort_index()  
    disease_summary[col] = {
        'Absent (0)': counts.get(0, 0),
        'Present (1)': counts.get(1, 0)
    }

disease_summary_df = pd.DataFrame(disease_summary).T
print(disease_summary_df)

          Absent (0)  Present (1)
calving        12144         4296
lameness       13224         3216
mastitis       15384         1056
oestrus         8496         7944


In [98]:
print(disease_present_df["cow"].unique())

[  156  1565  1624  1797  1919  2152  2155  2162  2164  2170  2182  2183
  2340  2395  2576  2581  2622  4102  4279  4372  4495  4716  5098  5104
  5128  5394  5541  5810  6601  6610  6612  6613  6621  6629  6633  6634
  6637  6638  6643  6646  6656  6664  6674  6675  6683  6686  6689  6690
  6693  6695  6699  6701  6714  6721  6750  7163  7600  8200  8605  8677
  9502  9601 10127 10567 42599 44432 46649 47875 47899 48133 48531 48555
 48589 49097 49378 49382 49407 49428 50249 50278 50351 50377 50398 50408
 50454 50478 50482 50493 50508 50514 50577 50590 50603 50627 50649 50653
 50678 50685 50738 50783 50802 53860 53884 53891 53923 53930 53961 53978
 53985 53992 54003 54027 54034 54065 54089 72650 72667 72681 72768 72775
 72799 72821 72845 72852 72890 72908 72915 72984 72991 73002 73026 73033
 73040 73064 73071 73095 73103 73110 73127 73172 73196 74441 74472 74489
 74528 74535 74559 74566 74573 74580 74629 74643 74650 74667 74681 74706
 74744 74799 74807 74814 74838 74852 74869 74876 74